# Phase 11 - repair the SQL that failed

19 of the 453 Phase 10 predictions did not execute. This notebook shows the
fine-tuned model each failing query together with the exact PostgreSQL error
and asks for a correction.

## Set these in the right-hand panel first

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** |
| Input 1 | the **training notebook's output** (the adapter) |
| Input 2 | **`text2sql-evalpack`** (for the schema) |
| Input 3 | **`text2sql-repairpack`** (the failures) |

Only 19 generations, so this is mostly model-loading time - expect around
15 minutes, nearly all of it the 16 GB download.

## What travels here

The question, the query that failed, and the database's error. **No gold**
**SQL and no gold result.** A repair loop that needs the answer in order to
fix a query would not work in production, which is the only place repair
matters.


## 1. Environment and inputs


In [ ]:
import os, sys, json, glob

import torch
assert torch.cuda.is_available(), 'No GPU - set Accelerator to GPU T4 x2'
p = torch.cuda.get_device_properties(0)
print(f'{p.name} | {p.total_memory/1024**3:.1f} GB | compute {p.major}.{p.minor}')

print()
print('--- /kaggle/input ---')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in sorted(files):
        print(' ', os.path.join(root, f))

def find(name):
    hits = [os.path.join(r, name)
            for r, _, fs in os.walk('/kaggle/input') if name in fs]
    if not hits:
        raise FileNotFoundError(
            f'{name} not found under /kaggle/input - check the Input panel')
    return hits[0]

ADAPTER_DIR = os.path.dirname(find('adapter_config.json'))
SCHEMA_F    = find('schema_context.txt')
REPAIRS_F   = find('repair_inputs.jsonl')
PROMPT_F    = find('repair_prompt_module.py')
MANIFEST_F  = find('manifest.json')

print()
print('adapter    :', ADAPTER_DIR)
print('repair pack:', os.path.dirname(REPAIRS_F))


## 2. Install dependencies


In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate 2>&1 | tail -3

import importlib.metadata as _md
for pkg in ['torch','transformers','peft','bitsandbytes','accelerate']:
    try:    print(f'{pkg:<15}{_md.version(pkg)}')
    except Exception: print(f'{pkg:<15}MISSING')


## 3. Verify the repair pack

The schema must still be the one the baseline and the fine-tune both saw, and
the repair prompt must be the version recorded with the results. Both are
recomputed from the shipped files rather than read out of the manifest.


In [ ]:
import hashlib, importlib.util

EXPECTED_SCHEMA_FP = 'd03619e711661bc5'
EXPECTED_REPAIR_FP = 'b66ccdd66e919abd'

spec = importlib.util.spec_from_file_location('repair_prompt_module', PROMPT_F)
rp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rp)
build_repair_messages = rp.build_repair_messages

SCHEMA = open(SCHEMA_F, encoding='utf-8').read()
schema_fp = hashlib.sha256(SCHEMA.encode('utf-8')).hexdigest()[:16]
repair_fp = rp.repair_prompt_fingerprint()

FAILURES = [json.loads(l) for l in open(REPAIRS_F, encoding='utf-8') if l.strip()]
MANIFEST = json.load(open(MANIFEST_F, encoding='utf-8'))

print(f'schema        {len(SCHEMA):,} chars  {schema_fp}  '
      f"{'MATCH' if schema_fp == EXPECTED_SCHEMA_FP else 'MISMATCH'}")
print(f'repair prompt {rp.REPAIR_PROMPT_VERSION}  {repair_fp}  '
      f"{'MATCH' if repair_fp == EXPECTED_REPAIR_FP else 'MISMATCH'}")
print(f'failures      {len(FAILURES)}')

assert schema_fp == EXPECTED_SCHEMA_FP, 'schema differs from the frozen baseline'
assert repair_fp == EXPECTED_REPAIR_FP, 'repair prompt differs from the export'
assert len(FAILURES) == MANIFEST['failures']['count']

extra = {k for f in FAILURES for k in f} - {'id','question','failed_sql','error','stage'}
assert not extra, f'repair pack carries unexpected fields: {extra}'
assert MANIFEST['failures']['gold_sql_included'] is False
print()
print('verified - no gold SQL present')
print()
for f in FAILURES[:3]:
    print('-' * 68)
    print('Q    :', f['question'][:90])
    print('SQL  :', f['failed_sql'][:110])
    print('ERROR:', f['error'][:90])


## 4. Load the base model in 4-bit and apply the adapter

Identical to the Phase 10 generation notebook, including the explicit chat
template load: the adapter's `tokenizer_config.json` carries no inline
template, and a tokenizer that silently falls back to a default would render
a prompt the model never saw.


In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_ID  = 'Qwen/Qwen3-8B'
REVISION = 'b968826d9c46'

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if not getattr(tok, 'chat_template', None):
    tok.chat_template = open(os.path.join(ADAPTER_DIR, 'chat_template.jinja'),
                             encoding='utf-8').read()
    print('chat template loaded explicitly from chat_template.jinja')
assert tok.chat_template and 'enable_thinking' in tok.chat_template, (
    'not the Qwen3 template the adapter was trained with')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'

t0 = time.perf_counter()
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID, revision=REVISION, quantization_config=bnb,
    device_map={'': 0}, torch_dtype=torch.float16, attn_implementation='sdpa')

n4 = sum(1 for m in base.modules() if m.__class__.__name__ == 'Linear4bit')
assert n4 > 0, 'model did NOT load in 4-bit'
print(f'4-bit load confirmed | {n4} Linear4bit modules | '
      f'VRAM {torch.cuda.memory_allocated()/1024**3:.2f} GB')

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
assert any('lora_' in n for n, _ in model.named_parameters()), 'adapter did not attach'
print(f'adapter attached | {time.perf_counter()-t0:.0f}s')


## 5. Render the repair prompt exactly as training did

Same guard as Phase 10, and it matters for the same reason: Qwen3's template
puts an empty `<think>` block before every assistant turn it was trained on,
and omitting it at inference produced unusable output the first time round.

The repair *content* is new to the model - Phase 9 trained on
question-to-SQL, never on error-to-correction - but the *format* must still
be the one it saw.


In [ ]:
SENTINEL = 'SELECT 1'


def render_repair_prompt(question, failed_sql, error):
    """The single definition of how a repair prompt is rendered."""
    return tok.apply_chat_template(
        build_repair_messages(question, SCHEMA, failed_sql, error),
        tokenize=False, add_generation_prompt=True, enable_thinking=False)


f0 = FAILURES[0]
demo = build_repair_messages(f0['question'], SCHEMA, f0['failed_sql'], f0['error'])
infer_text = render_repair_prompt(f0['question'], f0['failed_sql'], f0['error'])
train_text = tok.apply_chat_template(
    demo + [{'role': 'assistant', 'content': SENTINEL}], tokenize=False)

is_prefix = train_text.startswith(infer_text)
gap = train_text[len(infer_text):] if is_prefix else None
resumes_at_sql = bool(gap) and gap.startswith(SENTINEL)
print('inference prompt is a prefix of the training rendering:', is_prefix)
print('generation resumes exactly at the SQL              :', resumes_at_sql)
if is_prefix and not resumes_at_sql:
    print()
    print('!! the model would have to emit this first:',
          repr(gap.split(SENTINEL)[0]))

assert is_prefix, 'prompt rendering diverges from the training format'
assert resumes_at_sql, 'the model would have to generate filler before the SQL'

lens = [len(tok(render_repair_prompt(f['question'], f['failed_sql'], f['error']),
                add_special_tokens=False)['input_ids']) for f in FAILURES]
print(f'repair prompts: {len(lens)}, longest {max(lens):,} tokens')
assert max(lens) < 3500, 'a repair prompt is unexpectedly long'
print()
print('--- last 320 chars of the first repair prompt ---')
print(infer_text[-320:])


## 6. Generate repairs

Greedy, `max_new_tokens=512`, same as generation. Nineteen prompts is small
enough to run one at a time, which keeps the per-item latency honest rather
than averaged across a batch.


In [ ]:
REPAIRS = '/kaggle/working/repairs.jsonl'
MAX_NEW = 512
PROMPT_FORMAT = 'enable_thinking=False'

header = {'_header': True, 'schema_fingerprint': schema_fp,
          'repair_prompt_fingerprint': repair_fp,
          'repair_prompt_version': rp.REPAIR_PROMPT_VERSION,
          'base_model': BASE_ID, 'revision': REVISION,
          'adapter_dir': ADAPTER_DIR, 'decoding': 'greedy',
          'max_new_tokens': MAX_NEW, 'prompt_format': PROMPT_FORMAT,
          'gpu': p.name}

done = {}
if os.path.exists(REPAIRS):
    lines = [json.loads(l) for l in open(REPAIRS, encoding='utf-8') if l.strip()]
    old = next((r for r in lines if r.get('_header')), {})
    if old.get('prompt_format') != PROMPT_FORMAT:
        os.replace(REPAIRS, REPAIRS + '.stale')
        print('existing file used a different prompt format - starting fresh')
    else:
        done = {r['example_id']: r for r in lines if not r.get('_header')}
        print(f'resuming: {len(done)} already repaired')
if not os.path.exists(REPAIRS):
    with open(REPAIRS, 'w', encoding='utf-8') as fh:
        fh.write(json.dumps(header) + chr(10))

todo = [f for f in FAILURES if f['id'] not in done]
print(f'to repair: {len(todo)} of {len(FAILURES)}')

started = time.perf_counter()
fh = open(REPAIRS, 'a', encoding='utf-8')
for i, f in enumerate(todo, start=1):
    text = render_repair_prompt(f['question'], f['failed_sql'], f['error'])
    enc = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    n_in = enc['input_ids'].shape[1]
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    ms = (time.perf_counter() - t0) * 1000
    gen = out[0][n_in:]
    keep = [t for t in gen.tolist() if t != tok.pad_token_id]
    fh.write(json.dumps({
        'example_id': f['id'],
        'raw_output': tok.decode(gen, skip_special_tokens=True),
        'latency_ms': round(ms, 2), 'ok': True, 'error': None,
        'finish_reason': 'length' if len(keep) >= MAX_NEW else 'stop',
        'prompt_tokens': int(n_in), 'completion_tokens': len(keep),
    }, ensure_ascii=False) + chr(10))
    fh.flush()
    print(f'  {i:>3}/{len(todo)}  {ms:7.0f} ms  {f["id"]}', flush=True)
fh.close()
print()
print(f'done in {(time.perf_counter()-started)/60:.1f} minutes')


## 7. Look at what came back

Nothing is scored here - whether a repair actually fixes the query is decided
by running it against PostgreSQL on the laptop. This is only a check that the
model produced SQL rather than prose.


In [ ]:
rows = [json.loads(l) for l in open(REPAIRS, encoding='utf-8') if l.strip()]
reps = [r for r in rows if not r.get('_header')]
by_id = {f['id']: f for f in FAILURES}

n_sql = sum(1 for r in reps if 'SELECT' in r['raw_output'].upper())
n_cap = sum(1 for r in reps if r['finish_reason'] == 'length')
n_same = sum(1 for r in reps
             if r['raw_output'].strip() == by_id[r['example_id']]['failed_sql'].strip())

print('repairs           ', len(reps), 'of', len(FAILURES))
print('contain SELECT    ', n_sql)
print('hit token cap     ', n_cap)
print('identical to input', n_same, '(model changed nothing)')
print()
for r in reps[:5]:
    f = by_id[r['example_id']]
    print('=' * 68)
    print('Q      :', f['question'][:88])
    print('ERROR  :', f['error'][:88])
    print('BEFORE :', f['failed_sql'][:150])
    print('AFTER  :', r['raw_output'].strip()[:150])

print()
print('download repairs.jsonl from the Output tab')


---

## Next step, on the laptop

```powershell
env\Scripts\python.exe scripts/score_repair.py `
    --repairs "C:\Users\dell\Downloads\repairs.jsonl"
```

That executes each repaired query against PostgreSQL, folds the successes
into the Phase 10 predictions and rescores the whole 453 - producing
ablation configuration 5: fine-tuned + full schema + repair.
